# Reranker-Based RAG Experiment
I am building a small RAG pipeline to understand how
different retrieval methods affect the final context given to the language
model.

I compare **Vector Search, BM25, Hybrid Retrieval with RRF, and
Cross-Encoder Reranking** before generating the final answer with Gemini.

### Main Pipeline

**Question → Vector Search + BM25 → RRF → Candidate Documents → Reranker → Top-k Context → Gemini → Answer**

### What I Want to Test

- Vector Search vs BM25
- Individual retrieval vs Hybrid Retrieval
- Retrieval before vs after reranking
- Effect of different `candidate_k` and `final_k` values

I will use the actual outputs from my experiments to understand what changes
at each stage rather than assuming one method is always better.

## RAG Architecture

```text
        User Question
            │
      ├──────────────┐
      ▼              ▼
 Vector Search      BM25
   (FAISS)       (Keyword)
      │              │
      └──────┬───────┘
             ▼
        RRF Hybrid
        Retrieval
             │
             ▼
       Candidate Set
        candidate_k
             │
             ▼
     Cross-Encoder
        Reranker
             │
             ▼
        Final Top-k
          final_k
             │
             ▼
      Retrieved Context
             │
             ▼
       Gemini 2.5 Flash
             │
             ▼
         Final Answer

In [28]:
!pip -q install -U sentence-transformers faiss-cpu google-genai rank-bm25

In [29]:
import os
import re
import numpy as np
import faiss

from getpass import getpass

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from google import genai

## Connecting Gemini

The retrieval experiments do not require Gemini, but I need Gemini for the
last part of the notebook where I turn the retrieved context into an answer.

I am entering the API key interactively instead of putting the key directly
inside the notebook.

In [30]:
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client ready.")

Enter your Gemini API key: ··········
Gemini client ready.


## The small knowledge base

I am keeping the original knowledge base rather than replacing it with a new
dataset. It contains short documents about RAG, retrieval, embeddings,
BM25, APIs, error codes, query rewriting, contextual retrieval, and
rerankers.

Because the documents are short, this is mainly useful for understanding the
retrieval pipeline rather than measuring production-level RAG performance.

In [31]:
ocuments = [

    {
        "id": "doc1",
        "title": "RAG Introduction",
        "text": """
        Retrieval-Augmented Generation, commonly called RAG, combines
        information retrieval with large language models. Instead of relying
        only on the knowledge stored inside an LLM, RAG retrieves relevant
        information from an external knowledge base and provides that
        information to the language model as context. This allows an LLM
        to answer questions using external information.
        """
    },

    {
        "id": "doc2",
        "title": "Vector Databases",
        "text": """
        A vector database stores numerical representations of data called
        embeddings. These embeddings allow the system to perform semantic
        similarity searches. When a user asks a question, the question can
        be converted into an embedding and compared with document embeddings.
        Documents whose embeddings are most similar to the query can then
        be retrieved.
        """
    },

    {
        "id": "doc3",
        "title": "Embeddings",
        "text": """
        Embeddings are numerical vectors that represent the semantic meaning
        of text. Similar pieces of text tend to have embeddings that are close
        to each other in vector space. Embeddings are commonly used for
        semantic search, recommendation systems, clustering, document
        retrieval, and Retrieval-Augmented Generation systems.
        """
    },

    {
        "id": "doc4",
        "title": "RAG Pipeline",
        "text": """
        A typical RAG pipeline consists of document ingestion, text cleaning,
        text splitting, embedding generation, vector storage, retrieval,
        context construction, and language model generation. The retriever
        finds relevant chunks before the language model generates the answer.
        """
    },

    {
        "id": "doc5",
        "title": "Document Chunking",
        "text": """
        Chunking divides large documents into smaller pieces before embedding.
        Good chunking is important because excessively large chunks can contain
        irrelevant information while very small chunks may lose important
        context. Chunk size and overlap should be selected based on the
        document type and retrieval task.
        """
    },

    {
        "id": "doc6",
        "title": "Semantic Search",
        "text": """
        Semantic search retrieves information based on meaning rather than
        relying only on exact keyword matches. A query and documents are
        converted into embeddings, and similarity between the vectors is
        calculated. This allows semantic search to find relevant information
        even when the wording of the query differs from the wording in the
        document.
        """
    },

    {
        "id": "doc7",
        "title": "BM25 Keyword Search",
        "text": """
        BM25 is a lexical information retrieval algorithm. It ranks documents
        based on the occurrence of query terms and their importance within the
        document collection. BM25 is particularly useful for exact keywords,
        technical terminology, product names, identifiers, error codes, and
        other terms where exact matching is important.
        """
    },

    {
        "id": "doc8",
        "title": "Reranking",
        "text": """
        Reranking is a second-stage retrieval process. An initial retriever
        retrieves a larger set of candidate documents and a reranker then
        scores those candidates according to their relevance to the query.
        Reranking can improve precision by moving the most relevant documents
        toward the top of the results.
        """
    },

    {
        "id": "doc9",
        "title": "Hybrid Search",
        "text": """
        Hybrid search combines multiple retrieval methods, commonly semantic
        vector search and lexical keyword search. Vector search is useful for
        semantic meaning while keyword search is useful for exact terms.
        Combining the two approaches can improve retrieval robustness across
        different types of queries.
        """
    },

    {
        "id": "doc10",
        "title": "Authentication Errors",
        "text": """
        Authentication errors can occur when credentials are invalid,
        authentication tokens expire, or authorization policies reject a
        request.

        Authentication Error Codes:
        ERR-401 indicates an authentication failure.
        ERR-403 indicates that the user is authenticated but does not have
        permission to access a resource.

        Authentication tokens:
        Access tokens are used to authenticate requests. Refresh tokens are
        used to obtain new access tokens when the existing access token
        expires.
        """
    },

    {
        "id": "doc11",
        "title": "Product API",
        "text": """
        The Product API provides endpoints for creating, updating, deleting,
        and retrieving product records. Product records contain a product ID,
        name, price, inventory quantity, and category. The API uses JSON for
        request and response bodies.
        """
    },

    {
        "id": "doc12",
        "title": "Query Rewriting",
        "text": """
        Query rewriting transforms a user's original question into a clearer
        and more retrieval-friendly query. Query rewriting can expand missing
        context, resolve vague language, make important concepts explicit,
        remove unnecessary words, and produce terminology that better matches
        the knowledge base.
        """
    },

    {
        "id": "doc13",
        "title": "Query Expansion",
        "text": """
        Query expansion generates multiple alternative search queries from a
        single user query. The alternatives can use synonyms, related concepts,
        different terminology, or different formulations of the same question.
        Multiple searches can improve recall because relevant documents may use
        terminology different from the original user query.
        """
    },

    {
        "id": "doc14",
        "title": "Multi-Query Retrieval",
        "text": """
        Multi-query retrieval uses multiple search queries or perspectives for
        a single information need. Each query retrieves potentially different
        documents. The results are then combined using a ranking or fusion
        method. This approach improves retrieval recall and helps identify
        relevant information that may not be retrieved by a single query.
        """
    },

    {
        "id": "doc15",
        "title": "Query Decomposition",
        "text": """
        Query decomposition breaks a complex user question into smaller
        sub-questions. Each sub-question can be answered independently using
        retrieval. The individual evidence or intermediate answers can then
        be combined to answer the original complex question.
        """
    },

    {
        "id": "doc16",
        "title": "Contextual Retrieval",
        "text": """
        Contextual retrieval improves document chunks by adding information
        about where each chunk came from and what it means in the broader
        document. A contextualized chunk may include the document title,
        section, topic, and a short explanation of the chunk's relationship
        to the document. This additional context helps retrieval models
        understand otherwise ambiguous or incomplete chunks.
        """
    },

    {
        "id": "doc17",
        "title": "Reranker Models",
        "text": """
        A cross-encoder reranker takes a query and a candidate document
        together and calculates a relevance score. Unlike a bi-encoder,
        which independently embeds queries and documents, a cross-encoder can
        inspect the interaction between the query and candidate text more
        directly. Rerankers are commonly used as a second-stage retrieval
        component after an initial high-recall retriever.
        """
    },

    {
        "id": "doc18",
        "title": "Self-RAG",
        "text": """
        Self-RAG is a retrieval-augmented generation approach in which the
        language model evaluates whether retrieval is needed, whether
        retrieved evidence is relevant, and whether a generated answer is
        adequately supported. A Self-RAG system can decide to retrieve,
        critique retrieved context, generate an answer, evaluate that answer,
        and retry retrieval or generation when the evidence is insufficient.
        """
    }
]


## Chunking the documents

The original documents are already fairly small, but I still keep the same
chunking function because chunking is part of the RAG pipeline.

I used a chunk size of 70 words and an overlap of 15 words. The overlap means
that neighboring chunks can share some text instead of creating a hard
boundary between them.

For this small dataset, I expect chunking to have less impact than it would
on a collection of long documents. I am keeping the settings fixed so that
the retrieval comparisons below are easier to interpret.

In [32]:
def chunk_text(text, chunk_size=70, overlap=15):
    words = text.split()

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [34]:
chunks = []

for document in documents:
    document_chunks = chunk_text(
        document["text"],
        chunk_size=70,
        overlap=15
    )

    for chunk_number, chunk in enumerate(document_chunks):
        chunks.append({
            "chunk_id": f'{document["id"]}_chunk_{chunk_number}',
            "document_id": document["id"],
            "title": document["title"],
            "chunk_number": chunk_number,
            "text": chunk
        })

print("Documents:", len(documents))
print("Chunks:", len(chunks))

Documents: 17
Chunks: 17


In [35]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_dimension = (
    embedding_model.get_sentence_embedding_dimension()
)

print("Embedding dimension:", embedding_dimension)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


/tmp/ipykernel_4039/2176670758.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [36]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (17, 384)


In [37]:
vector_index = faiss.IndexFlatIP(
    embedding_dimension
)

vector_index.add(embeddings)

print("Vectors stored in FAISS:", vector_index.ntotal)

Vectors stored in FAISS: 17


In [38]:
vector_index = faiss.IndexFlatIP(
    embedding_dimension
)

vector_index.add(embeddings)

print("Vectors stored in FAISS:", vector_index.ntotal)

Vectors stored in FAISS: 17


In [39]:
def tokenize(text):
    return re.findall(
        r"\b\w+\b",
        text.lower()
    )


tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(
    tokenized_chunks
)

print("BM25 index ready.")

BM25 index ready.


In [40]:
def vector_search(query, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = vector_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx == -1:
            continue

        result = chunks[idx].copy()
        result["vector_score"] = float(score)
        result["vector_rank"] = rank

        results.append(result)

    return results

In [41]:
def bm25_search(query, top_k=10):
    scores = bm25.get_scores(
        tokenize(query)
    )

    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(
        indices,
        start=1
    ):
        result = chunks[idx].copy()

        result["bm25_score"] = float(
            scores[idx]
        )

        result["bm25_rank"] = rank

        results.append(result)

    return results

In [42]:
experiment_queries = [
    "How does a reranker improve RAG retrieval?",
    "What does ERR-401 mean?"
]

for query in experiment_queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)

    vector_results = vector_search(
        query,
        top_k=5
    )

    bm25_results = bm25_search(
        query,
        top_k=5
    )

    print("\nVector search:")

    for i, result in enumerate(
        vector_results,
        start=1
    ):
        print(
            f"{i}. {result['title']} "
            f"(score={result['vector_score']:.4f})"
        )

    print("\nBM25:")

    for i, result in enumerate(
        bm25_results,
        start=1
    ):
        print(
            f"{i}. {result['title']} "
            f"(score={result['bm25_score']:.4f})"
        )


QUERY: How does a reranker improve RAG retrieval?

Vector search:
1. Reranking (score=0.4828)
2. RAG Pipeline (score=0.4809)
3. RAG Introduction (score=0.4307)
4. Reranker Models (score=0.4049)
5. Multi-Query Retrieval (score=0.2653)

BM25:
1. Reranking (score=4.5957)
2. RAG Pipeline (score=3.2036)
3. RAG Introduction (score=3.1989)
4. Reranker Models (score=3.1163)
5. Authentication Errors (score=2.6503)

QUERY: What does ERR-401 mean?

Vector search:
1. Authentication Errors (score=0.5523)
2. BM25 Keyword Search (score=0.1556)
3. Reranking (score=0.1294)
4. RAG Pipeline (score=0.1201)
5. Reranker Models (score=0.1099)

BM25:
1. Authentication Errors (score=7.0511)
2. Contextual Retrieval (score=2.2117)
3. Reranker Models (score=0.0000)
4. Multi-Query Retrieval (score=0.0000)
5. Query Decomposition (score=0.0000)


## Combining the two retrieval methods

The vector and BM25 scores are not directly comparable, so I am not adding
their raw scores.

Instead, I use Reciprocal Rank Fusion.

The basic idea is simple: a document gets credit for appearing near the top
of either ranking, and a document appearing near the top of both rankings
gets more combined support.

I use the same RRF constant of 60 from my original implementation.

In [43]:
def reciprocal_rank_fusion(result_lists, k=60):
    fused = {}

    for results in result_lists:

        for rank, result in enumerate(
            results,
            start=1
        ):
            chunk_id = result["chunk_id"]

            if chunk_id not in fused:
                fused[chunk_id] = {
                    "chunk": result,
                    "rrf_score": 0.0
                }

            fused[chunk_id]["rrf_score"] += (
                1.0 / (k + rank)
            )

    ranked = sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    final_results = []

    for item in ranked:
        result = item["chunk"].copy()
        result["rrf_score"] = item["rrf_score"]
        final_results.append(result)

    return final_results

In [44]:
def hybrid_search(query, candidate_k=20):

    vector_results = vector_search(
        query,
        top_k=candidate_k
    )

    bm25_results = bm25_search(
        query,
        top_k=candidate_k
    )

    fused_results = reciprocal_rank_fusion(
        [
            vector_results,
            bm25_results
        ]
    )

    return fused_results[:candidate_k]

In [45]:
query = "How does contextual retrieval improve RAG?"

vector_results = vector_search(
    query,
    top_k=5
)

bm25_results = bm25_search(
    query,
    top_k=5
)

hybrid_results = hybrid_search(
    query,
    candidate_k=5
)

print("QUERY:")
print(query)

print("\nVECTOR SEARCH")
for i, result in enumerate(
    vector_results,
    start=1
):
    print(
        f"{i}. {result['title']}"
    )

print("\nBM25")
for i, result in enumerate(
    bm25_results,
    start=1
):
    print(
        f"{i}. {result['title']}"
    )

print("\nHYBRID / RRF")
for i, result in enumerate(
    hybrid_results,
    start=1
):
    print(
        f"{i}. {result['title']} "
        f"(RRF={result['rrf_score']:.5f})"
    )

QUERY:
How does contextual retrieval improve RAG?

VECTOR SEARCH
1. RAG Pipeline
2. RAG Introduction
3. Contextual Retrieval
4. Reranker Models
5. Multi-Query Retrieval

BM25
1. RAG Introduction
2. Contextual Retrieval
3. RAG Pipeline
4. Hybrid Search
5. Authentication Errors

HYBRID / RRF
1. RAG Introduction (RRF=0.03252)
2. RAG Pipeline (RRF=0.03227)
3. Contextual Retrieval (RRF=0.03200)
4. Reranker Models (RRF=0.01562)
5. Hybrid Search (RRF=0.01562)


In [46]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross-Encoder loaded.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder loaded.


In [47]:
def rerank_documents(
    query,
    documents,
    top_k=5
):

    if not documents:
        return []

    pairs = [
        [query, document["text"]]
        for document in documents
    ]

    scores = reranker.predict(
        pairs
    )

    reranked = []

    for document, score in zip(
        documents,
        scores
    ):
        result = document.copy()
        result["reranker_score"] = float(score)
        reranked.append(result)

    reranked.sort(
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return reranked[:top_k]

In [48]:
query = "How does a reranker improve RAG retrieval?"

candidate_k = 10
final_k = 5

candidates = hybrid_search(
    query,
    candidate_k=candidate_k
)

reranked_results = rerank_documents(
    query,
    candidates,
    top_k=final_k
)

print("QUERY:")
print(query)

print("\nBEFORE RERANKING")

for i, result in enumerate(
    candidates,
    start=1
):
    print(
        f"{i}. {result['title']} "
        f"(RRF={result['rrf_score']:.5f})"
    )

print("\nAFTER RERANKING")

for i, result in enumerate(
    reranked_results,
    start=1
):
    print(
        f"{i}. {result['title']} "
        f"(reranker={result['reranker_score']:.4f})"
    )

QUERY:
How does a reranker improve RAG retrieval?

BEFORE RERANKING
1. Reranking (RRF=0.03279)
2. RAG Pipeline (RRF=0.03226)
3. RAG Introduction (RRF=0.03175)
4. Reranker Models (RRF=0.03125)
5. Multi-Query Retrieval (RRF=0.03009)
6. Contextual Retrieval (RRF=0.02964)
7. Query Expansion (RRF=0.02921)
8. Authentication Errors (RRF=0.01538)
9. Hybrid Search (RRF=0.01515)
10. Document Chunking (RRF=0.01493)

AFTER RERANKING
1. Reranking (reranker=5.3805)
2. Reranker Models (reranker=1.3694)
3. RAG Introduction (reranker=0.0844)
4. RAG Pipeline (reranker=-3.1862)
5. Contextual Retrieval (reranker=-3.5087)


In [49]:
old_positions = {
    result["chunk_id"]: position
    for position, result in enumerate(
        candidates,
        start=1
    )
}

print("POSITION CHANGES")

for new_position, result in enumerate(
    reranked_results,
    start=1
):
    old_position = old_positions.get(
        result["chunk_id"]
    )

    print(
        f"{result['title']}: "
        f"{old_position} -> {new_position}"
    )

POSITION CHANGES
Reranking: 1 -> 1
Reranker Models: 4 -> 2
RAG Introduction: 3 -> 3
RAG Pipeline: 2 -> 4
Contextual Retrieval: 6 -> 5


In [50]:
query = "How does contextual retrieval improve RAG?"

for candidate_k in [5, 10, 15, 20]:

    candidates = hybrid_search(
        query,
        candidate_k=candidate_k
    )

    reranked = rerank_documents(
        query,
        candidates,
        top_k=min(5, len(candidates))
    )

    print("\n" + "=" * 60)
    print("candidate_k =", candidate_k)

    print("\nHybrid candidates:")
    for i, result in enumerate(
        candidates,
        start=1
    ):
        print(
            f"{i}. {result['title']}"
        )

    print("\nReranked results:")
    for i, result in enumerate(
        reranked,
        start=1
    ):
        print(
            f"{i}. {result['title']} "
            f"(score={result['reranker_score']:.4f})"
        )


candidate_k = 5

Hybrid candidates:
1. RAG Introduction
2. RAG Pipeline
3. Contextual Retrieval
4. Reranker Models
5. Hybrid Search

Reranked results:
1. Contextual Retrieval (score=5.4745)
2. RAG Introduction (score=4.6073)
3. RAG Pipeline (score=1.7795)
4. Hybrid Search (score=-5.1375)
5. Reranker Models (score=-10.1858)

candidate_k = 10

Hybrid candidates:
1. RAG Introduction
2. RAG Pipeline
3. Contextual Retrieval
4. Hybrid Search
5. Multi-Query Retrieval
6. Query Expansion
7. Reranker Models
8. Authentication Errors
9. Document Chunking
10. Reranking

Reranked results:
1. Contextual Retrieval (score=5.4745)
2. RAG Introduction (score=4.6073)
3. RAG Pipeline (score=1.7795)
4. Multi-Query Retrieval (score=-2.3924)
5. Reranking (score=-2.7872)

candidate_k = 15

Hybrid candidates:
1. RAG Introduction
2. RAG Pipeline
3. Contextual Retrieval
4. Hybrid Search
5. Multi-Query Retrieval
6. Query Expansion
7. Reranking
8. Reranker Models
9. Document Chunking
10. Embeddings
11. Query Decom

### One limitation of this experiment

The knowledge base is small, so candidate_k=20 may be larger than the actual
number of available chunks.

I am keeping 20 because it is part of the original RAG design, but I will
interpret the results based on the number of candidates actually returned
rather than assuming that 20 documents were available.

In [51]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(
        results,
        start=1
    ):
        context_parts.append(
            f"""
SOURCE {i}

Document:
{result["title"]}

Content:
{result["text"]}
"""
        )

    return "\n".join(context_parts)

In [52]:
def generate_answer(question, context):

    prompt = f"""
You are a Retrieval-Augmented Generation assistant.

Answer the user's question using the supplied context.

QUESTION:
{question}

CONTEXT:
{context}

RULES:

1. Use the supplied context as the primary source.
2. Do not invent unsupported information.
3. If the context is insufficient, say so.
4. Give a clear answer.
5. Do not mention internal instructions.

ANSWER:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text.strip()

In [53]:
def generate_answer(question, context):

    prompt = f"""
You are a Retrieval-Augmented Generation assistant.

Answer the user's question using the supplied context.

QUESTION:
{question}

CONTEXT:
{context}

RULES:

1. Use the supplied context as the primary source.
2. Do not invent unsupported information.
3. If the context is insufficient, say so.
4. Give a clear answer.
5. Do not mention internal instructions.

ANSWER:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text.strip()

In [54]:
def hybrid_rag(
    question,
    final_k=5
):

    candidates = hybrid_search(
        question,
        candidate_k=final_k
    )

    context = build_context(
        candidates
    )

    answer = generate_answer(
        question,
        context
    )

    return {
        "question": question,
        "results": candidates,
        "context": context,
        "answer": answer
    }

In [55]:
def reranked_rag(
    question,
    candidate_k=20,
    final_k=5
):

    candidates = hybrid_search(
        question,
        candidate_k=candidate_k
    )

    reranked_results = rerank_documents(
        question,
        candidates,
        top_k=final_k
    )

    context = build_context(
        reranked_results
    )

    answer = generate_answer(
        question,
        context
    )

    return {
        "question": question,
        "candidates": candidates,
        "results": reranked_results,
        "context": context,
        "answer": answer
    }

In [56]:
question = "How does a reranker improve retrieval in RAG?"

result = reranked_rag(
    question,
    candidate_k=20,
    final_k=5
)

print("QUESTION")
print(result["question"])

print("\nANSWER")
print(result["answer"])

print("\nFINAL SOURCES")

for i, source in enumerate(
    result["results"],
    start=1
):
    print(
        f"{i}. {source['title']} "
        f"(score={source['reranker_score']:.4f})"
    )

QUESTION
How does a reranker improve retrieval in RAG?

ANSWER
In RAG, a reranker improves retrieval by acting as a second-stage process after an initial retriever has gathered a larger set of candidate documents. The reranker scores these candidate documents based on their relevance to the query, specifically by inspecting the interaction between the query and the candidate text more directly (as described for a cross-encoder reranker). This scoring process improves precision by moving the most relevant documents to the top of the results, ensuring that the best context is provided to the language model.

FINAL SOURCES
1. Reranking (score=4.8432)
2. Reranker Models (score=0.9133)
3. RAG Introduction (score=0.9014)
4. RAG Pipeline (score=-2.5933)
5. Contextual Retrieval (score=-3.4921)
